# 07 · Motors and Mixing

### Recap & why now
Notebook 06's simulator accepted a **wrench** — a total thrust and three torques — and
no quadcopter on earth has that input. A flight controller commands four numbers: how
hard each motor should push.

The conversion is a $4\times4$ matrix, which is easy. What is not easy, and what this
notebook is really about, is what happens when the conversion asks for a thrust the
hardware cannot produce.

### Learning objectives
1. Build the **mixing matrix** from the X-configuration geometry, not by copying it.
2. Explain where roll, pitch and yaw authority each come from.
3. **Invert** the mixer to find the motor thrusts a desired wrench needs.
4. Apply the limits $0 \le T_i \le T_{\max}$ and measure what saturation delivers.
5. Verify the six motor patterns that produce the six basic motions.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

def quat_from_rotmat(R):
    """Rotation matrix -> quaternion. Four branches, so we never divide by a small number."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)

def axis_angle_to_quat(axis, angle):
    """Build a quaternion from 'rotate by `angle` about `axis`' — the geometric reading."""
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(axis*np.sin(angle/2))])

def quat_rotate(q, v):
    """Rotate v from the body frame into the world frame, using the sandwich product."""
    return quat_multiply(quat_multiply(q, np.array([0.0, *v])), quat_conjugate(q))[1:]

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

# === Mixing and 6-DOF dynamics, from Notebooks 06-07 =====================

POS, VEL, QUAT, OMEGA = slice(0, 3), slice(3, 6), slice(6, 10), slice(10, 13)
MIX = np.vstack([np.ones(4), MOTOR_POS[:, 1], -MOTOR_POS[:, 0], -SPIN*PARAMS["d"]])

def motor_mixer(total_thrust, torques, p=PARAMS):
    """Desired wrench -> four motor thrusts, clipped to what the hardware can do."""
    T4 = np.linalg.solve(MIX, np.array([total_thrust, *torques], float))
    return np.clip(T4, p["T_min"], p["T_max"])     # A propeller cannot pull, nor push forever.

def quad_dynamics(state, motor_thrusts, p=PARAMS, f_ext=np.zeros(3)):
    """x_dot for the 13-state quadcopter, driven by four motor thrusts."""
    q = quat_normalize(state[QUAT]); w = state[OMEGA]
    T, tx, ty, tz = MIX @ np.asarray(motor_thrusts, float)          # Geometry does its job here.
    v_dot = (quat_to_rotmat(q) @ np.array([0.0, 0.0, T])            # Thrust, body -> world.
             + np.array([0.0, 0.0, -p["m"]*g]) + f_ext)/p["m"]      # Gravity, ENU, plus any push.
    q_dot = 0.5*quat_multiply(q, np.array([0.0, *w]))               # Notebook 05's kinematics.
    w_dot = np.linalg.solve(p["I"], np.array([tx, ty, tz]) - np.cross(w, p["I"] @ w))
    return np.concatenate([state[VEL], v_dot, q_dot, w_dot])

def rk4_step(state, motor_thrusts, dt, p=PARAMS, f_ext=np.zeros(3)):
    """One RK4 step, followed by the renormalisation Notebook 05 insisted on."""
    k1 = quad_dynamics(state, motor_thrusts, p, f_ext)
    k2 = quad_dynamics(state + 0.5*dt*k1, motor_thrusts, p, f_ext)
    k3 = quad_dynamics(state + 0.5*dt*k2, motor_thrusts, p, f_ext)
    k4 = quad_dynamics(state + dt*k3, motor_thrusts, p, f_ext)
    s = state + dt/6*(k1 + 2*k2 + 2*k3 + k4)
    s[QUAT] = quat_normalize(s[QUAT])
    return s

def make_state(p=(0, 0, 0), v=(0, 0, 0), q=(1, 0, 0, 0), w=(0, 0, 0)):
    """Assemble the 13-element state vector."""
    return np.concatenate([p, v, q, w]).astype(float)

def simulate(command, T_end=4.0, dt=0.005, s0=None, p=PARAMS, f_ext=lambda t: np.zeros(3)):
    """Fly the drone. `command(t, state)` returns four motor thrusts in newtons."""
    s = make_state() if s0 is None else np.array(s0, float)
    ts, xs, ms = [0.0], [s.copy()], []
    for k in range(int(round(T_end/dt))):
        T4 = np.clip(np.asarray(command(k*dt, s), float), p["T_min"], p["T_max"])
        s = rk4_step(s, T4, dt, p, f_ext(k*dt))
        ts.append((k+1)*dt); xs.append(s.copy()); ms.append(T4)
    return np.array(ts), np.array(xs), np.array(ms)

T_HOVER = PARAMS["m"]*g                            # Total thrust that exactly cancels weight.
HOVER_EACH = T_HOVER/4                             # ...split over four identical motors.
print("model ready — hover needs %.4f N total, %.4f N per motor" % (T_HOVER, HOVER_EACH))

## 1 · Where each authority comes from

A rotor thrust $T_i$ at body position $r_i$ produces a torque $\tau_i = r_i \times F_i$
with $F_i = [0, 0, T_i]$:

$$r \times \begin{bmatrix}0\\0\\T\end{bmatrix} = \begin{bmatrix} r_y T \\ -r_x T \\ 0 \end{bmatrix}$$

So a motor's $y$ position gives it **roll** authority and its $x$ position gives it
**pitch** authority — and the cross product produces a zero in the third slot, which
means thrust can never twist the drone about $z$.

Yaw comes from **drag** instead: the air resists each spinning rotor and pushes back on
the airframe, opposite to the spin, in proportion to thrust.

In [ ]:
print("  motor   position [m]                roll arm    pitch arm    spin")
for i, (mp, sp) in enumerate(zip(MOTOR_POS, SPIN)):
    print("   M%d    %-26s %+9.4f %+12.4f %6s" %
          (i+1, np.round(mp, 4), mp[1], -mp[0], "CCW" if sp > 0 else "CW"))

print("\nmixing matrix M (rows: total thrust, roll, pitch, yaw):")
print(np.round(MIX, 5))
print("\ncondition number %.1f — well behaved, so the inverse below is trustworthy." % np.linalg.cond(MIX))
print("\nNotice the size of the last row: %.3f against %.3f in the roll row. Yaw authority is" %
      (PARAMS["d"], abs(MOTOR_POS[0, 1])))
print("about %.0f times weaker, because it is levered by drag rather than by an arm." %
      (abs(MOTOR_POS[0, 1])/PARAMS["d"]))

## 2 · Four thrusts in, one wrench out

$$\begin{bmatrix} T \\ \tau_x \\ \tau_y \\ \tau_z \end{bmatrix} = M
\begin{bmatrix} T_1 \\ T_2 \\ T_3 \\ T_4 \end{bmatrix}$$

Read the rows as four questions. Everybody lifts; the left pair rolls against the right
pair; the front pair pitches against the rear pair; alternate motors yaw against each
other. Each pattern below moves exactly one entry of the wrench.

In [ ]:
hov = HOVER_EACH
print("  pattern                    motor thrusts [N]            -> [T, tau_x, tau_y, tau_z]")
patterns = [("all equal (hover)      ", [hov]*4),
            ("all up 10%             ", [hov*1.1]*4),
            ("left pair up  (M2, M3) ", [hov-0.1, hov+0.1, hov+0.1, hov-0.1]),
            ("rear pair up  (M3, M4) ", [hov-0.1, hov-0.1, hov+0.1, hov+0.1]),
            ("CW pair up    (M1, M3) ", [hov+0.1, hov-0.1, hov+0.1, hov-0.1])]
for name, T4 in patterns:
    print("  %s %-28s %s" % (name, np.round(T4, 3), np.round(MIX @ np.array(T4), 4)))

print("\nEach pattern moves ONE entry and leaves the others at zero. That decoupling is a")
print("property of the X layout, not a coincidence — and it is what makes M invertible.")

## 3 · Going backwards, and hitting the limits

A controller thinks in wrench. The motors need thrusts, so we solve $M T = u$ — using
`np.linalg.solve` rather than forming an explicit inverse, a habit that pays on larger
systems.

Then physics intervenes: $0 \le T_i \le T_{\max}$. When the solved thrusts fall outside
that band we clip them, and the moment we do, **the drone stops receiving the wrench
that was requested**. Clipping is a silent change of plan.

In [ ]:
def delivered(total_thrust, torques, p=PARAMS):
    """The wrench the airframe really feels, once the clipping has happened."""
    return MIX @ motor_mixer(total_thrust, torques, p)

print("  request [T, tau]                motor thrusts [N]        delivered wrench")
for T, tau in [(T_HOVER, [0.05, 0, 0]), (T_HOVER, [0.4, 0, 0]),
               (4*PARAMS["T_max"], [0, 0, 0]), (4*PARAMS["T_max"], [0.4, 0, 0])]:
    T4 = motor_mixer(T, tau, PARAMS)
    flag = " <-- SATURATED" if np.any(np.isclose(T4, PARAMS["T_max"])) or np.any(np.isclose(T4, 0)) else ""
    print("  %-32s %-24s %s%s" % (np.round([T, *tau], 2), np.round(T4, 3),
                                  np.round(delivered(T, tau), 3), flag))

got = delivered(4*PARAMS["T_max"], [0.4, 0, 0])
print("\nThe last row asked for maximum climb AND a hard roll. The motors that should have")
print("pushed harder were already flat out, so only the descending pair could move: the roll")
print("arrived at %.0f%% of the request, and the thrust came up short too." % (100*got[1]/0.4))
print("A saturated quadcopter does not fail loudly — it quietly does something ELSE.")

## 4 · Six checks

Six motor patterns, six expected motions. Every row is a physical claim the mixing
matrix had to get right, and any one of them failing would mean a sign error somewhere
in the geometry.

In [ ]:
checks = [("hover, all equal   ", [hov]*4,                                 "stays put"),
          ("all motors +10%    ", [hov*1.1]*4,                             "climbs"),
          ("all motors -10%    ", [hov*0.9]*4,                             "sinks"),
          ("left pair stronger ", [hov-0.05, hov+0.05, hov+0.05, hov-0.05], "rolls about +x only"),
          ("rear pair stronger ", [hov-0.05, hov-0.05, hov+0.05, hov+0.05], "pitches about +y only"),
          ("CW pair stronger   ", [hov+0.05, hov-0.05, hov+0.05, hov-0.05], "yaws about +z only")]

print("  pattern              position after 0.5 s        roll/pitch/yaw [deg]     expected")
for name, T4, expect in checks:
    t, X, M = simulate(lambda t_, s_: T4, T_end=0.5)          # Short, so the angles stay readable.
    print("  %s %-26s %-24s %s" %
          (name, np.round(X[-1, POS], 3), np.round(np.degrees(quat_to_euler(X[-1, QUAT])), 2), expect))

print("\nOnly the intended axis moves in every row, and the yawing drone stays exactly over its")
print("start point — the claim Notebook 01 made about yaw, now confirmed by the dynamics.")
print("Note how small the imbalances are: 0.05 N is 2%% of one motor's hover thrust, and it")
print("rolls the drone noticeably in half a second. Attitude authority is cheap.")

## 🧪 Try it yourself

**E1.** Raise **one** motor and leave the other three at hover. Which entries of the
wrench change, and why can no quadcopter manoeuvre use a single motor?

**E2.** Find the largest yaw torque this vehicle can deliver while hovering, and compare
it with the largest roll torque. Does the ratio match the geometry?

In [ ]:
# --- Solution E1 ---
T4 = np.array([hov + 0.4, hov, hov, hov])
print("E1: raising M1 alone gives wrench", np.round(MIX @ T4, 4))
print("    THREE things change at once. M1 sits at (+a, -a), so it adds lift, generates negative")
print("    roll AND negative pitch, and contributes yaw through its drag. One motor cannot be an")
print("    actuator on its own — a quadcopter has four inputs and four outputs, and every clean")
print("    manoeuvre is a COMBINATION. That is exactly why we invert a matrix instead of")
print("    reasoning motor by motor.")

# --- Solution E2 ---
def saturation_limit(axis, hi, n=800):
    """Largest torque on `axis` the mixer can still deliver exactly, while hovering."""
    req = np.linspace(0, hi, n)
    got = []
    for r in req:
        tau = np.zeros(3); tau[axis] = r
        got.append(delivered(T_HOVER, tau)[axis+1])
    short = np.where(np.array(got) < req - 1e-6)[0]
    return req[short[0]] if len(short) else np.nan

yaw_lim, roll_lim = saturation_limit(2, 0.4), saturation_limit(0, 2.5)
print("\nE2: largest deliverable yaw torque at hover  %.4f N m" % yaw_lim)
print("    largest deliverable roll torque at hover %.4f N m" % roll_lim)
print("    ratio %.1f, against the geometric prediction a/d = %.4f/%.4f = %.1f ✔" %
      (roll_lim/yaw_lim, abs(MOTOR_POS[0, 1]), PARAMS["d"], abs(MOTOR_POS[0, 1])/PARAMS["d"]))
print("    Roll and pitch are levered by the arm; yaw only has aerodynamic drag to work with.")
print("    Every multirotor pilot has felt this as sluggish yaw, and this is the number behind it.")

## 🚁 Mini-project: the four-motor discovery lab

Rather than being told which combinations do what, work it out. Give the rig a pattern
of offsets from hover and it flies the drone for half a second and reports what
happened. Try to predict each row before you read it — especially the last two.

In [ ]:
def try_pattern(offsets, label, T_end=0.5):
    """Fly a motor pattern and report the wrench and the motion it produced."""
    T4 = np.full(4, hov) + np.asarray(offsets, float)
    t, X, _ = simulate(lambda t_, s_: T4, T_end=T_end)
    rpy = np.degrees(quat_to_euler(X[-1, QUAT]))
    print("  %-24s wrench [%6.2f %6.3f %6.3f %6.3f]   moved %s   turned %s" %
          (label, *np.round(MIX @ T4, 3), np.round(X[-1, POS], 2), np.round(rpy, 1)))

print("  offsets are added to the hover thrust of %.3f N on each motor:\n" % hov)
try_pattern([ 0.1,  0.1,  0.1,  0.1], "1 all four up")
try_pattern([-0.1,  0.1,  0.1, -0.1], "2 left pair up")
try_pattern([-0.1, -0.1,  0.1,  0.1], "3 rear pair up")
try_pattern([ 0.1, -0.1,  0.1, -0.1], "4 CW pair up together")
try_pattern([ 0.2,  0.0,  0.0,  0.0], "5 M1 alone")
try_pattern([ 0.15, 0.0, -0.15, 0.0], "6 M1 up, M3 down")

print("\nPattern 6 is the instructive one, and probably not what you predicted. M1 and M3 are on")
print("the SAME diagonal and spin the SAME way, so raising one while lowering the other leaves")
print("total lift untouched AND cancels the yaw exactly — their drag torques are equal and")
print("opposite. What survives is a tilt along the other diagonal. Compare pattern 4, where the")
print("same pair moves TOGETHER: there the lift changes cancel and the drag torques add, giving")
print("yaw. Same two motors, opposite result, decided entirely by the sign pattern.")

# ---- and one of them, flown ---------------------------------------------
T4_yaw = np.full(4, hov) + np.array([0.1, -0.1, 0.1, -0.1])   # Pattern 4: the pure yaw command.
t, X, M = simulate(lambda t_, s_: T4_yaw, T_end=3.0)
step = 12
fig = plt.figure(figsize=(6.4, 5.0))
ax = fig.add_subplot(111, projection="3d")

def frame(j):
    ax.clear()
    k = step*j
    draw_quad(ax, X[k, POS], X[k, QUAT], scale=2.6, thrusts=M[min(k, len(M)-1)])
    set_3d(ax, (-1, 1), (-1, 1), (-0.6, 1.4))
    rpy_k = np.degrees(quat_to_euler(X[k, QUAT]))
    ax.set_title("CW pair up: yaw %6.1f°   roll %4.1f°   pitch %4.1f°" % (rpy_k[2], rpy_k[0], rpy_k[1]),
                 fontsize=10)
    ax.view_init(elev=26, azim=-60)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(X)//step, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** The mixer is where control theory meets sheet metal.
> Change to a hexacopter and $M$ becomes $4\times6$ — more motors than commands, so the
> inverse becomes a least-squares problem and the aircraft gains the ability to lose a
> motor and keep flying. Tilt the rotors and $M$ gains columns that produce sideways
> force directly. Every one of those is a change to $M$ and nothing upstream, which is
> why the mixer is a separate box in every autopilot ever written.

**Where next.** We can command the drone the way real hardware expects. Notebook 08
builds the fast inner loops that keep it upright.